# Task 6: Bidirectional LSTM cell mechanics from scratch

In [1]:
import torch

torch.manual_seed(0)


In [2]:
def lstm_cell(x, h_prev, c_prev, params):
    Wi,Ui,bi, Wf,Uf,bf, Wo,Uo,bo, Wc,Uc,bc = params
    i = torch.sigmoid(x @ Wi + h_prev @ Ui + bi)
    f = torch.sigmoid(x @ Wf + h_prev @ Uf + bf)
    o = torch.sigmoid(x @ Wo + h_prev @ Uo + bo)
    c_tilde = torch.tanh(x @ Wc + h_prev @ Uc + bc)
    c = f*c_prev + i*c_tilde
    h = o*torch.tanh(c)
    return h, c

def init_lstm_params(input_dim, hidden_dim):
    params = []
    for _ in range(4):
        params += [torch.randn(input_dim, hidden_dim)*0.1,
                   torch.randn(hidden_dim, hidden_dim)*0.1,
                   torch.zeros(hidden_dim)]
    return params


In [3]:
def run_direction(X, params, hidden_dim):
    batch, seq_len, input_dim = X.shape
    h = torch.zeros(batch, hidden_dim)
    c = torch.zeros(batch, hidden_dim)
    outputs = []
    for t in range(seq_len):
        h, c = lstm_cell(X[:,t,:], h, c, params)
        outputs.append(h.unsqueeze(1))
    return torch.cat(outputs, dim=1)

def bilstm_forward(X, hidden_dim=8):
    input_dim = X.shape[-1]
    fwd_params = init_lstm_params(input_dim, hidden_dim)
    bwd_params = init_lstm_params(input_dim, hidden_dim)
    fwd_out = run_direction(X, fwd_params, hidden_dim)
    bwd_out = run_direction(torch.flip(X, dims=[1]), bwd_params, hidden_dim)
    bwd_out = torch.flip(bwd_out, dims=[1])
    return torch.cat([fwd_out, bwd_out], dim=-1)


In [4]:
X = torch.randn(4, 10, 6)
out = bilstm_forward(X)
print(out.shape)


torch.Size([4, 10, 16])
